# 01 — Feature Engineering

**Input:** `data/jobstreet_cleaned_final.csv` (output of the cleaning phase)
**Outputs:** `data/model_table.csv` (training table for `02_models.ipynb`) plus the dashboard artifacts `models/skill_lists.joblib`, `models/job_title_suggestions.joblib` and `models/skill_stats.joblib`

This notebook prepares the modelling dataset in eight steps:

1. **Filter** to rows with a trustworthy salary target (`salary_usable_flag == 1`, RM500–30,000/month).
2. **Extract `experience_years`** from the job description text with regular expressions.
3. **Extract `edu_level`** (required education, SPM → postgraduate) from the same text.
4. **Flag ~75 skills** (hard + soft) in the description using a transparent keyword dictionary.
5. **Assemble** the model table (features + target).
6. **Save** the model table and the skill lists.
7. **Job title suggestions** — the search-style dropdown + category auto-fill used by the dashboard.
8. **Skill statistics** — how often each skill appears in higher-paying ads per role, the evidence behind the dashboard's career advice.

TF-IDF of the job title and one-hot encoding of the categorical columns are *deliberately not done here* — they are fitted inside the sklearn `Pipeline` in `02_models.ipynb`, so the exact same transformation is applied later when the dashboard makes predictions on new input.

In [1]:
# Imports: pandas/numpy for data handling, re for regex extraction,
# os + joblib for saving artifacts.
import pandas as pd
import numpy as np
import re
import os
import joblib

# Load the cleaned dataset produced by the data cleaning notebook
df = pd.read_csv("../data/jobstreet_cleaned_final.csv")
print("Loaded:", f"{df.shape[0]:,}", "rows x", df.shape[1], "columns")

# Confirm the columns this notebook depends on are present
needed = ["salary_usable_flag", "salary_monthly_final", "descriptions_clean",
          "job_title", "category", "state_clean", "type_clean"]
missing = [c for c in needed if c not in df.columns]
print("All required columns present:", len(missing) == 0)

Loaded: 69,024 rows x 29 columns
All required columns present: True


## Step 1 — Filter to usable salary rows

The target must **never be imputed** — a made-up salary would teach the model made-up patterns. Only rows where the cleaning phase marked the advertised salary as usable are kept. On top of that, monthly salaries outside **RM500–30,000** are removed: values in that range are almost certainly parsing errors or non-monthly figures (daily rates, annual packages) that slipped through cleaning.

In [2]:
# Keep only rows whose salary can be trusted as a training target
before = len(df)
df = df[df["salary_usable_flag"] == 1].copy()
print(f"After salary_usable_flag == 1 : {len(df):,} rows (dropped {before - len(df):,})")

# Remove implausible monthly salaries (locked decision: RM500 - RM30,000)
before = len(df)
df = df[df["salary_monthly_final"].between(500, 30000)].copy()
print(f"After RM500 <= salary <= RM30,000 : {len(df):,} rows (dropped {before - len(df):,})")

# Sanity check against known dataset facts (median ~RM3,750, IQR RM3,000-5,000)
q = df["salary_monthly_final"].quantile([0.25, 0.5, 0.75])
print(f"\nTarget sanity check -> 25%: RM{q[0.25]:,.0f} | median: RM{q[0.5]:,.0f} | 75%: RM{q[0.75]:,.0f}")

After salary_usable_flag == 1 : 31,468 rows (dropped 37,556)
After RM500 <= salary <= RM30,000 : 31,406 rows (dropped 62)

Target sanity check -> 25%: RM3,000 | median: RM3,750 | 75%: RM5,000


## Step 2 — Extract years of experience

Required experience is one of the strongest wage drivers, but JobStreet has no structured field for it — it only appears inside the free-text description (*"minimum 3 years"*, *"2-4 years"*, *"5+ years"*). Two regular expressions extract it:

- a **range** like "2-4 years" → take the **minimum** (that is the entry requirement);
- a **single figure** like "3 years" or "5+ years" → take the number.

Decisions (and why):
- a years figure only counts when the word **"experience"** (or Malay **"pengalaman"**) appears within **60 characters** of it — without this check, company-age phrases like *"established more than 40 years ago"* were matched as requirements (the substring check also catches *"experienced"* and *"berpengalaman"*);
- when several figures appear in one ad, the **smallest** is used — the conservative reading of the entry requirement;
- values are **capped at 20** years, protecting against any nonsense matches that slip through;
- when nothing is stated, `experience_years = 0` **and** `has_experience_req = 0` — the flag lets the model tell *"no experience required"* apart from *"not mentioned"*.

In [3]:
# Pattern 1: a range like "2-4 years" / "2 to 4 years" (also tolerates "2-4+ years")
RANGE_PATTERN = re.compile(r"(\d{1,2})\s*(?:-|–|to)\s*(\d{1,2})\s*\+?\s*(?:years?|yrs?)\b")
# Pattern 2: a single figure like "3 years" / "5+ yrs"
SINGLE_PATTERN = re.compile(r"(\d{1,2})\s*\+?\s*(?:years?|yrs?)\b")
# A years match only counts when this word appears close to it. Substring match on
# purpose: it also catches "experienced" and Malay "pengalaman"/"berpengalaman".
CONTEXT_PATTERN = re.compile(r"experience|pengalaman")
CONTEXT_WINDOW = 60   # characters checked on each side of the years match

def has_experience_context(text, match):
    """True if 'experience'/'pengalaman' appears within 60 chars of the match."""
    start = max(match.start() - CONTEXT_WINDOW, 0)
    end = min(match.end() + CONTEXT_WINDOW, len(text))
    return CONTEXT_PATTERN.search(text[start:end]) is not None

def extract_experience(text):
    """Return (experience_years, has_experience_req) for one job description."""
    if not isinstance(text, str) or not text.strip():
        return 0, 0
    text = text.lower()
    candidates = []
    for m in RANGE_PATTERN.finditer(text):
        if has_experience_context(text, m):
            candidates.append(min(int(m.group(1)), int(m.group(2))))  # min of a range
    for m in SINGLE_PATTERN.finditer(text):
        if has_experience_context(text, m):
            candidates.append(int(m.group(1)))
    if not candidates:
        return 0, 0                                   # no requirement stated
    years = min(min(candidates), 20)                  # smallest mention, capped at 20
    return years, 1

# Demonstrate on hand-made examples before touching real data.
# The company-age sentence must give (0, 0) - that is the point of the context check.
tests = [
    "Minimum 3 years experience in accounting",
    "2-4 years of relevant experience required",
    "at least 5+ yrs experience in sales",
    "Fresh graduates are encouraged to apply",
    "our company was established more than 40 years ago",
    "pengalaman kerja sekurang-kurangnya 2 years dalam jualan",
]
for t in tests:
    print(f"{t!r:60} -> {extract_experience(t)}")

'Minimum 3 years experience in accounting'                   -> (3, 1)
'2-4 years of relevant experience required'                  -> (2, 1)
'at least 5+ yrs experience in sales'                        -> (5, 1)
'Fresh graduates are encouraged to apply'                    -> (0, 0)
'our company was established more than 40 years ago'         -> (0, 0)
'pengalaman kerja sekurang-kurangnya 2 years dalam jualan'   -> (2, 1)


In [4]:
# Apply the extractor to every description in the filtered dataset
extracted = df["descriptions_clean"].apply(extract_experience)
df["experience_years"] = [e[0] for e in extracted]
df["has_experience_req"] = [e[1] for e in extracted]

n_req = df["has_experience_req"].sum()
print(f"Ads stating an experience requirement: {n_req:,} of {len(df):,} ({n_req / len(df):.1%})")
print("\nexperience_years distribution:")
print(df["experience_years"].describe().round(2))
print("\nMost common values:")
print(df["experience_years"].value_counts().head(10).sort_index())
print(f"\nRows at the 20-year cap: {(df['experience_years'] == 20).sum():,}")

# Show real matches so the extraction is verifiable (report evidence)
def first_accepted_match(text):
    """First years match that passes the experience-context check."""
    for pattern in (RANGE_PATTERN, SINGLE_PATTERN):
        for m in pattern.finditer(text):
            if has_experience_context(text, m):
                return m
    return None

print("\nExamples from real descriptions (accepted matches):")
sample = df[df["has_experience_req"] == 1].sample(4, random_state=42)
for _, row in sample.iterrows():
    text = row["descriptions_clean"].lower()
    m = first_accepted_match(text)
    start, end = max(m.start() - 40, 0), min(m.end() + 40, len(text))
    snippet = text[start:end].replace("\n", " ")
    print(f'  "...{snippet}..."  ->  {row["experience_years"]} years')

# The known false positive from the first version of this notebook: an ad whose
# "more than 40 years" refers to COMPANY AGE, not required experience. With the
# context check it must now be rejected (expected: 0 years, flag 0).
mask = df["descriptions_clean"].str.lower().str.contains("industry more than 40 years", na=False)
row = df[mask].iloc[0]
text = row["descriptions_clean"].lower()
idx = text.find("more than 40 years")
snippet = text[max(idx - 40, 0): idx + 60].replace("\n", " ")
print("\nCompany-age mention (false positive in the first version):")
print(f'  "...{snippet}..."  ->  {row["experience_years"]} years, '
      f'has_experience_req = {row["has_experience_req"]}')

Ads stating an experience requirement: 16,307 of 31,406 (51.9%)

experience_years distribution:
count    31406.00
mean         1.47
std          2.26
min          0.00
25%          0.00
50%          1.00
75%          2.00
max         20.00
Name: experience_years, dtype: float64

Most common values:
experience_years
0     15192
1      4484
2      4623
3      3638
4       462
5      2095
6       137
8       223
10      235
20      126
Name: count, dtype: int64

Rows at the 20-year cap: 126

Examples from real descriptions (accepted matches):
  "...onal requirement for seo team: required 1 – 3 years of experience in seo , sem or web devel..."  ->  1 years
  "...ong kong and china country and min 1 or 2 years relevant working experiences is added a..."  ->  2 years
  "...uates are encouraged to apply *at least 1-2 year(s) of working experience in related fie..."  ->  1 years
  ".... fast learner, mature and independent. 1-2 years working experience tasks & responsibili..."  ->  1 years



Company-age mention (false positive in the first version):
  "... malaysia bathroom and kitchen industry more than 40 years and we have a strong existing clients pro..."  ->  0 years, has_experience_req = 0


## Step 3 — Extract education requirement

Education is the other big requirement job ads state only in free text (*"Candidates must possess at least a Bachelor's Degree..."*, *"Minimum SPM"*). Keyword patterns turn it into an **ordinal level** — higher requirement, higher number:

| `edu_level` | Meaning | Keywords |
|---|---|---|
| 0 | not stated | — |
| 1 | SPM / secondary | spm, stpm, o-level |
| 2 | Diploma | diploma |
| 3 | Bachelor's degree | bachelor, degree, ijazah |
| 4 | Postgraduate | master's degree, masters in, mba, phd, doctorate, postgraduate |

Decisions (and why):

- when an ad mentions several levels (*"Diploma or Degree in any field"*), the **minimum** is taken — the same conservative entry-requirement reading as the experience extraction;
- the word **"degree"** gets two hand-written exclusions: a digit right before it is an angle (*"360 degree feedback"*), and *"degree of"* is the abstract sense (*"a high degree of accuracy"*) — neither is education;
- bare **"master"** is never matched (it fires on *"Scrum Master"*); only unambiguous forms like *"master's degree"* count;
- unlike years-of-experience, **no context window** is needed — words like *"diploma"* and *"spm"* are unambiguous by themselves;
- `edu_level = 0` **and** `has_edu_req = 0` when nothing is stated, mirroring `has_experience_req`.

*This is an approved extension of locked modelling decision 3 (2026-07-15): `edu_level` and `has_edu_req` join the feature set, and all models are retrained in `02_models.ipynb`.*

In [5]:
# Education level patterns. Unlike bare year numbers, words like "diploma" or
# "spm" are unambiguous on their own, so no context window is needed here —
# only the word "degree" gets hand-written exclusion checks (see below).
SECONDARY_PATTERN = re.compile(r"\bspm\b|\bstpm\b|\bo[\s\-]levels?\b")
DIPLOMA_PATTERN   = re.compile(r"\bdiplomas?\b")
BACHELOR_PATTERN  = re.compile(r"\bbachelors?\b|\bijazah\b")   # ijazah = Malay for degree
DEGREE_PATTERN    = re.compile(r"\bdegrees?\b")                # only with the checks below
POSTGRAD_PATTERN  = re.compile(                                # never bare "master" —
    r"\bmaster(?:'s|s)?\s+degree\b|\bmasters\s+in\b"           # it fires on "Scrum Master"
    r"|\bmba\b|\bph\.?d\b|\bdoctorate\b|\bpostgraduates?\b")

EDU_LEVEL_NAMES = {0: "Not stated", 1: "SPM / secondary", 2: "Diploma",
                   3: "Bachelor's degree", 4: "Postgraduate"}

def is_academic_degree(text, m):
    """Keep only education uses of the word 'degree'.

    Rejects angles ("360 degree feedback" — a digit right before the word)
    and the abstract sense ("a high degree of accuracy" — followed by ' of').
    """
    prefix = text[max(m.start() - 4, 0):m.start()]
    if any(ch.isdigit() for ch in prefix):
        return False
    return not text[m.end():m.end() + 4].startswith(" of")

def extract_education(text):
    """Return (edu_level, has_edu_req) for one job description.

    Level = MINIMUM education mentioned — "Diploma or Degree" is an entry
    requirement of Diploma. Same conservative reading as the min-of-range
    rule in the experience extraction.
    """
    if not isinstance(text, str) or not text.strip():
        return 0, 0
    text = text.lower()
    levels = []
    if SECONDARY_PATTERN.search(text):
        levels.append(1)
    if DIPLOMA_PATTERN.search(text):
        levels.append(2)
    if BACHELOR_PATTERN.search(text) or any(
            is_academic_degree(text, m) for m in DEGREE_PATTERN.finditer(text)):
        levels.append(3)
    if POSTGRAD_PATTERN.search(text):
        levels.append(4)
    if not levels:
        return 0, 0
    return min(levels), 1

# Demonstrate on hand-made examples before touching real data. The two
# non-education uses of "degree" must both give (0, 0).
tests = [
    "Candidates must possess at least a Bachelor's Degree in Accounting",
    "Minimum SPM with 2 years working experience",
    "Diploma or Degree in any related field",
    "MBA or Masters in Finance preferred",
    "360 degree feedback culture in our company",
    "a high degree of accuracy is expected",
    "ijazah sarjana muda dalam bidang perakaunan",
    "Fresh graduates are encouraged to apply",
]
for t in tests:
    level, flag = extract_education(t)
    print(f"{t!r:68} -> ({level}, {flag})  {EDU_LEVEL_NAMES[level]}")

"Candidates must possess at least a Bachelor's Degree in Accounting" -> (3, 1)  Bachelor's degree
'Minimum SPM with 2 years working experience'                        -> (1, 1)  SPM / secondary
'Diploma or Degree in any related field'                             -> (2, 1)  Diploma
'MBA or Masters in Finance preferred'                                -> (4, 1)  Postgraduate
'360 degree feedback culture in our company'                         -> (0, 0)  Not stated
'a high degree of accuracy is expected'                              -> (0, 0)  Not stated
'ijazah sarjana muda dalam bidang perakaunan'                        -> (3, 1)  Bachelor's degree
'Fresh graduates are encouraged to apply'                            -> (0, 0)  Not stated


In [6]:
# Apply the education extractor to every description in the filtered dataset
extracted_edu = df["descriptions_clean"].apply(extract_education)
df["edu_level"] = [e[0] for e in extracted_edu]
df["has_edu_req"] = [e[1] for e in extracted_edu]

n_edu = df["has_edu_req"].sum()
print(f"Ads stating an education requirement: {n_edu:,} of {len(df):,} ({n_edu / len(df):.1%})")

print("\nedu_level distribution:")
for level, count in df["edu_level"].value_counts().sort_index().items():
    print(f"  {level} = {EDU_LEVEL_NAMES[level]:18}: {count:>6,} ads ({count / len(df):5.1%})")

# Education should carry a real salary signal — median salary per level is the
# sanity check (and a report figure)
print("\nMedian salary by required education level:")
med = df.groupby("edu_level")["salary_monthly_final"].agg(["count", "median"])
for level, row in med.iterrows():
    print(f"  {EDU_LEVEL_NAMES[level]:18}: median RM{row['median']:>7,.0f}  (n={int(row['count']):,})")

# Show real matched snippets so the extraction is verifiable (report evidence)
def first_edu_match(text):
    """First education keyword match in a description (for evidence snippets)."""
    for pattern in (SECONDARY_PATTERN, DIPLOMA_PATTERN, BACHELOR_PATTERN, POSTGRAD_PATTERN):
        m = pattern.search(text)
        if m:
            return m
    for m in DEGREE_PATTERN.finditer(text):
        if is_academic_degree(text, m):
            return m
    return None

print("\nExamples from real descriptions (accepted matches):")
sample = df[df["has_edu_req"] == 1].sample(5, random_state=42)
for _, row in sample.iterrows():
    text = row["descriptions_clean"].lower()
    m = first_edu_match(text)
    start, end = max(m.start() - 45, 0), min(m.end() + 45, len(text))
    snippet = text[start:end].replace("\n", " ")
    print(f'  "...{snippet}..."  ->  {EDU_LEVEL_NAMES[row["edu_level"]]}')

Ads stating an education requirement: 21,949 of 31,406 (69.9%)

edu_level distribution:
  0 = Not stated        :  9,457 ads (30.1%)
  1 = SPM / secondary   :  2,881 ads ( 9.2%)
  2 = Diploma           : 10,434 ads (33.2%)
  3 = Bachelor's degree :  8,609 ads (27.4%)
  4 = Postgraduate      :     25 ads ( 0.1%)

Median salary by required education level:
  Not stated        : median RM  3,750  (n=9,457)
  SPM / secondary   : median RM  3,000  (n=2,881)
  Diploma           : median RM  3,750  (n=10,434)
  Bachelor's degree : median RM  4,750  (n=8,609)
  Postgraduate      : median RM  4,750  (n=25)

Examples from real descriptions (accepted matches):
  "...tion and growth. qualifications requirements bachelor’s degree in marketing, communications, or a ..."  ->  Bachelor's degree
  "...succeed in this role, you're required to be: diploma & degree or equivalent; must be enrolled in ..."  ->  Diploma
  "...date must possess at least bachelor's degree/diploma in any field. be able to commu

## Step 4 — Skill flags from a keyword dictionary

Skills influence wages and are exactly the lever a job-seeker can act on — the dashboard's improvement tips re-predict with extra skills toggled on. A hand-picked dictionary (**50 hard + 25 soft skills**) is matched with **word-boundary regular expressions** on the lowercased description. This approach is fully transparent: every flag traces back to one keyword, which is easy to defend at the viva (unlike embedding-based methods).

Matching details:
- word boundaries stop false positives — `java` must not fire inside `javascript`;
- spaces in a skill also match hyphens, so *"problem solving"* and *"problem-solving"* both count;
- an optional trailing "s" is allowed (*"presentations"*, *"apis"*);
- skills containing `+ # .` (like `c++`, `.net`) get hand-written patterns, because the default `\b` boundary breaks next to punctuation;
- `skill_count` (total skills mentioned) is added as a simple proxy for how demanding a role is.

In [7]:
# The skill dictionary. This exact list is saved to models/ and reused as the
# options of the dashboard's skill multiselect, so the model and the app can
# never disagree about which skills exist.
HARD_SKILLS = [
    "python", "java", "javascript", "typescript", "c++", "c#", "php", "sql",
    "html", "css", "react", "angular", "vue", "node.js", ".net",
    "excel", "power bi", "tableau", "sap", "autocad", "solidworks",
    "photoshop", "illustrator", "canva", "wordpress", "seo",
    "google analytics", "salesforce", "oracle", "mysql", "mongodb",
    "aws", "azure", "docker", "kubernetes", "linux", "git", "api",
    "machine learning", "data analysis", "accounting", "bookkeeping",
    "payroll", "taxation", "audit", "quickbooks", "digital marketing",
    "social media marketing", "copywriting", "customer relationship management",
]
SOFT_SKILLS = [
    "communication", "teamwork", "leadership", "problem solving",
    "time management", "critical thinking", "creativity", "adaptability",
    "attention to detail", "multitasking", "negotiation", "presentation",
    "customer service", "interpersonal", "analytical", "organizational",
    "decision making", "self motivated", "independent", "fast learner",
    "work under pressure", "collaboration", "initiative", "flexibility",
    "positive attitude",
]
print(f"{len(HARD_SKILLS)} hard skills + {len(SOFT_SKILLS)} soft skills = "
      f"{len(HARD_SKILLS) + len(SOFT_SKILLS)} total")

50 hard skills + 25 soft skills = 75 total


In [8]:
# Hand-written patterns for skills where the default word boundary fails
# (punctuation) or where common word forms would otherwise be missed.
SPECIAL_PATTERNS = {
    "c++":     r"\bc\+\+",
    "c#":      r"\bc#",
    ".net":    r"\.net\b",
    "node.js": r"\bnode\.?js\b",                       # node.js / nodejs
    "multitasking": r"\bmulti[\s\-]?tasking\b",        # multitasking / multi-tasking
    "organizational": r"\borgani[sz]ational\b",        # US + UK spelling
    "audit": r"\baudit(?:ing|ors?|s)?\b",              # audit / auditing / auditor
    "independent": r"\bindependent(?:ly)?\b",          # "work independently"
    "work under pressure": r"\bwork(?:ing)?\s+under\s+pressure\b",
}

def make_pattern(skill):
    """Word-boundary regex for a skill; spaces also match hyphens, plural 's' allowed."""
    if skill in SPECIAL_PATTERNS:
        return SPECIAL_PATTERNS[skill]
    words = [re.escape(w) for w in skill.split()]
    return r"\b" + r"[\s\-]+".join(words) + r"s?\b"

def make_column_name(skill):
    """Safe DataFrame column name, e.g. 'c++' -> 'skill_c_plus_plus'."""
    s = (skill.replace("c++", "c_plus_plus").replace("c#", "c_sharp")
              .replace(".net", "dot_net").replace("node.js", "node_js"))
    return "skill_" + re.sub(r"[^a-z0-9]+", "_", s).strip("_")

ALL_SKILLS = HARD_SKILLS + SOFT_SKILLS
skill_columns = {skill: make_column_name(skill) for skill in ALL_SKILLS}

# Show a few generated patterns as evidence they behave sensibly
for s in ["java", "c++", "problem solving", "audit", "work under pressure"]:
    print(f"{s:20} -> {skill_columns[s]:28} pattern: {make_pattern(s)}")

java                 -> skill_java                   pattern: \bjavas?\b
c++                  -> skill_c_plus_plus            pattern: \bc\+\+
problem solving      -> skill_problem_solving        pattern: \bproblem[\s\-]+solvings?\b
audit                -> skill_audit                  pattern: \baudit(?:ing|ors?|s)?\b
work under pressure  -> skill_work_under_pressure    pattern: \bwork(?:ing)?\s+under\s+pressure\b


In [9]:
# Flag every skill in every description (one str.contains scan per skill)
desc_lower = df["descriptions_clean"].fillna("").str.lower()
for skill in ALL_SKILLS:
    df[skill_columns[skill]] = desc_lower.str.contains(
        make_pattern(skill), regex=True).astype(int)

flag_cols = list(skill_columns.values())
df["skill_count"] = df[flag_cols].sum(axis=1)

# Top 20 most demanded skills (report figure)
prevalence = df[flag_cols].mean().sort_values(ascending=False)
top20 = pd.DataFrame({
    "skill": [c.replace("skill_", "").replace("_", " ") for c in prevalence.head(20).index],
    "job_ads": df[prevalence.head(20).index].sum().astype(int).values,
    "share_of_ads": (prevalence.head(20) * 100).round(1).astype(str).values + " %",
})
print("Top 20 skills by share of job ads:")
print(top20.to_string(index=False))

print("\nskill_count per ad:")
print(df["skill_count"].describe().round(2))

never_matched = [c for c in flag_cols if df[c].sum() == 0]
print("\nSkills never matched:", never_matched if never_matched else "none")

Top 20 skills by share of job ads:
              skill  job_ads share_of_ads
      communication    16108       51.3 %
        independent     8401       26.7 %
      interpersonal     7327       23.3 %
         accounting     5909       18.8 %
              excel     5759       18.3 %
    problem solving     5251       16.7 %
              audit     4886       15.6 %
         analytical     4728       15.1 %
         initiative     3745       11.9 %
   customer service     3729       11.9 %
     organizational     3393       10.8 %
       presentation     3302       10.5 %
attention to detail     3283       10.5 %
         leadership     2921        9.3 %
     self motivated     2493        7.9 %
    time management     2178        6.9 %
        negotiation     2171        6.9 %
      collaboration     1999        6.4 %
            payroll     1834        5.8 %
                sql     1486        4.7 %

skill_count per ad:
count    31406.00
mean         3.67
std          2.80
min     

## Step 5 — Assemble the model table

One row per usable job ad, containing exactly the features of the locked modelling design plus the target. `job_title`, `category`, `state_clean` and `type_clean` stay **raw** — TF-IDF and one-hot encoding happen inside the Pipeline in `02_models.ipynb`. The engineered columns (`experience_years`, `has_experience_req`, `edu_level`, `has_edu_req`, the skill flags and `skill_count`) pass through unchanged. The target `salary_monthly_final` stays in RM; the log-transform also happens in `02_models`.

In [10]:
# Final columns: 4 raw columns for the Pipeline + engineered features + target
feature_cols = (["job_title", "category", "state_clean", "type_clean",
                 "experience_years", "has_experience_req", "edu_level", "has_edu_req"]
                + flag_cols + ["skill_count"])
model_table = df[feature_cols + ["salary_monthly_final"]].reset_index(drop=True)

print("Model table:", f"{model_table.shape[0]:,}", "rows x", model_table.shape[1], "columns")
print("\nColumn dtypes:")
print(model_table.dtypes.value_counts())
print("\nMissing values in model table:", int(model_table.isna().sum().sum()))
print("\nFirst 5 rows (skill flag columns not shown):")
preview_cols = ["job_title", "category", "state_clean", "type_clean",
                "experience_years", "has_experience_req", "edu_level", "has_edu_req",
                "skill_count", "salary_monthly_final"]
print(model_table[preview_cols].head().to_string(index=False))

Model table: 31,406 rows x 85 columns

Column dtypes:
int64      80
str         4
float64     1
Name: count, dtype: int64

Missing values in model table: 0

First 5 rows (skill flag columns not shown):
                   job_title                             category  state_clean type_clean  experience_years  has_experience_req  edu_level  has_edu_req  skill_count  salary_monthly_final
Account Executive/ Assistant                           Accounting     Selangor  Full time                 0                   0          2            1            0                3000.0
            Service Engineer                          Engineering     Selangor  Full time                 2                   1          2            1            3                3250.0
        Purchasing Executive Manufacturing, Transport & Logistics     Selangor  Full time                 1                   1          1            1            3                3150.0
          Accounts Executive                      

## Step 6 — Save the artifacts

- `data/model_table.csv` — the training table `02_models.ipynb` starts from (kept in `data/`, which is git-ignored).
- `models/skill_lists.joblib` — hard/soft skill lists plus the display-name → column-name mapping. The dashboard loads **only** `models/` artifacts (never the CSV), and its skill multiselect is built from this file, so app and model always agree on the skill vocabulary.

Both files are reloaded immediately after saving to prove they round-trip.

In [11]:
os.makedirs("../models", exist_ok=True)

# 1. Model table for 02_models
table_path = "../data/model_table.csv"
model_table.to_csv(table_path, index=False)

# 2. Skill lists for 02_models and the dashboard multiselect
skills_path = "../models/skill_lists.joblib"
joblib.dump({
    "hard_skills": HARD_SKILLS,
    "soft_skills": SOFT_SKILLS,
    "skill_columns": skill_columns,     # display name -> model column name
}, skills_path)

for p in [table_path, skills_path]:
    print(f"Saved {p} ({os.path.getsize(p) / 1024:,.0f} KB)")

# Round-trip check: reload both artifacts as 02_models / the dashboard will
reloaded = joblib.load(skills_path)
check = pd.read_csv(table_path)
print("\nReload check -> skills:", len(reloaded["hard_skills"]), "hard /",
      len(reloaded["soft_skills"]), "soft | model table:", check.shape)
print("Feature engineering complete.")

Saved ../data/model_table.csv (7,382 KB)
Saved ../models/skill_lists.joblib (3 KB)



Reload check -> skills: 50 hard / 25 soft | model table: (31406, 85)
Feature engineering complete.


## Step 7 — Job title suggestions for the dashboard

The dashboard's job-title box should behave like a search field: typing "data" suggests
*data scientist*, *data entry* and so on, and picking a suggestion auto-fills the job
category. Because the dashboard is only allowed to load `models/` artifacts (never the
CSV), the suggestion list is built here and saved as one more artifact.

Decisions (and why):
- Titles are grouped **case-insensitively** — the first run of this cell showed
  "Account Executive" (459 ads) and "ACCOUNT EXECUTIVE" (128 ads) as two separate
  suggestions. Each group is displayed with its **most common casing**.
- Only titles appearing in **at least 3 job ads** are suggested — one-off titles are
  usually noisy or overly specific ("Admin cum Account Assistant (Mandarin Speaker)"),
  and a shorter list keeps the dropdown responsive. Free-text input stays possible in
  the app, so nothing is lost for users with rare titles.
- Suggestions are **ordered by frequency**, so the most common titles surface first.
- Each kept title maps to its **most common category** in the data — that is what the
  category auto-fill uses. The mapping is keyed by the **lowercased** title, so the
  dashboard can also auto-fill when the user types a known title in different casing.

In [12]:
# Group titles case-insensitively: strip whitespace, count by lowercased form
titles_df = pd.DataFrame({
    "original": model_table["job_title"].str.strip(),
    "category": model_table["category"],
})
titles_df["lower"] = titles_df["original"].str.lower()

group_counts = titles_df["lower"].value_counts()
kept_counts = group_counts[group_counts >= 3]
print(f"Unique job titles (case-insensitive): {len(group_counts):,}")
print(f"Titles appearing >= 3 times (kept as suggestions): {len(kept_counts):,} "
      f"(covering {kept_counts.sum():,} of {len(model_table):,} ads, "
      f"{kept_counts.sum() / len(model_table):.1%})")

kept_rows = titles_df[titles_df["lower"].isin(kept_counts.index)]
# Display form of each group = its most common original casing
display_form = (kept_rows.groupby("lower")["original"]
                .agg(lambda s: s.value_counts().idxmax()))
# Auto-fill target = most common category per group, keyed by LOWERCASED title so
# the dashboard can also look up titles the user typed in different casing
title_to_category = (kept_rows.groupby("lower")["category"]
                     .agg(lambda s: s.mode()[0])
                     .to_dict())

# Suggestions ordered by frequency (most common titles first in the dropdown)
suggestion_titles = [display_form[t] for t in kept_counts.index]

print("\nTop 10 suggested titles (with auto-fill category):")
for t in suggestion_titles[:10]:
    lower = t.lower()
    print(f"  {t:35} ({kept_counts[lower]:>4} ads) -> {title_to_category[lower]}")

# The user-story example: typing 'data' should surface data-related titles
data_titles = [t for t in suggestion_titles if "data" in t.lower()]
print(f"\nSuggestions containing 'data' ({len(data_titles)} total, first 8):")
for t in data_titles[:8]:
    print(f"  {t:35} -> {title_to_category[t.lower()]}")

# Save as one more dashboard artifact and prove it round-trips
suggestions_path = "../models/job_title_suggestions.joblib"
joblib.dump({
    "titles": suggestion_titles,            # display casing, ordered by frequency
    "title_to_category": title_to_category, # LOWERCASED title -> most common category
}, suggestions_path)
reloaded_sugg = joblib.load(suggestions_path)
print(f"\nSaved {suggestions_path} ({os.path.getsize(suggestions_path) / 1024:,.0f} KB) | "
      f"reload check: {len(reloaded_sugg['titles']):,} titles, "
      f"{len(reloaded_sugg['title_to_category']):,} category mappings")

Unique job titles (case-insensitive): 17,134
Titles appearing >= 3 times (kept as suggestions): 1,257 (covering 13,929 of 31,406 ads, 44.4%)



Top 10 suggested titles (with auto-fill category):
  Account Executive                   ( 592 ads) -> Accounting
  Sales Executive                     ( 379 ads) -> Sales
  Accounts Executive                  ( 358 ads) -> Accounting
  Admin Assistant                     ( 284 ads) -> Administration & Office Support
  Account Assistant                   ( 222 ads) -> Accounting
  Marketing Executive                 ( 222 ads) -> Marketing & Communications
  Admin Executive                     ( 206 ads) -> Administration & Office Support
  Project Engineer                    ( 167 ads) -> Engineering
  Finance Executive                   ( 147 ads) -> Accounting
  Business Development Executive      ( 142 ads) -> Sales

Suggestions containing 'data' (10 total, first 8):
  Data Analyst                        -> Information & Communication Technology
  Database Administrator              -> Information & Communication Technology
  Data Scientist                      -> Science & Techno

## Step 8 — Skill statistics for the career advice

The dashboard's *Career improvement opportunities* section recommends skills to learn. A
positive model uplift alone is **not** enough to recommend a skill — the model happily
reports that "kubernetes" would raise an HR manager's predicted salary, even though no
HR job actually asks for it. The recommendation therefore needs dataset evidence: *how
common is this skill in higher-paying ads for this role?* Because the dashboard is only
allowed to load `models/` artifacts, that evidence is precomputed here.

Decisions (and why):
- **"Higher-paying"** = ads paying **above the median salary of their own group**. Comparing
  within the group (HR ads against HR ads) is what makes the signal role-specific.
- Two levels of grouping: every **job category** (broad, always has enough ads) and every
  **job title with at least 30 usable ads** (specific — "higher-paying HR Manager ads" —
  but only where a percentage is statistically meaningful). The dashboard uses the title
  stats when the entered title is in the artifact and falls back to the category otherwise.
- Per group and skill, two numbers are stored: the **count** of higher-paying ads mentioning
  the skill (the "based on N advertisements" evidence) and the **share** of higher-paying
  ads that is ("common in 61% of ...").

In [13]:
# Title-level stats need enough ads for a percentage to mean something
MIN_TITLE_ADS = 30

def skill_stats_for(group_df):
    """Skill evidence for one group of ads (one job title or one category).

    'Higher-paying' = ads above the group's own median salary. Returns
    {"n_high": <higher-paying ads>, "skills": {skill: (count, share)}} with
    count = higher-paying ads mentioning the skill, share = count / n_high.
    Skills that never appear in the group's higher-paying ads are left out.
    """
    median = group_df["salary_monthly_final"].median()
    high = group_df[group_df["salary_monthly_final"] > median]
    if high.empty:
        return None   # degenerate group (all salaries identical)
    stats = {}
    for skill in ALL_SKILLS:
        n_with = int(high[skill_columns[skill]].sum())
        if n_with >= 1:
            stats[skill] = (n_with, n_with / len(high))
    return {"n_high": len(high), "skills": stats}

# Category level: all 30 categories are big enough
by_category = {}
for cat, group in model_table.groupby("category"):
    stats = skill_stats_for(group)
    if stats:
        by_category[cat] = stats

# Title level: only titles with >= MIN_TITLE_ADS usable ads (grouped
# case-insensitively, keyed lowercase — same convention as the suggestions)
title_lower = model_table["job_title"].str.strip().str.lower()
by_title = {}
for title, group in model_table.groupby(title_lower):
    if len(group) >= MIN_TITLE_ADS:
        stats = skill_stats_for(group)
        if stats:
            by_title[title] = stats

print(f"Category groups: {len(by_category)} | title groups (>= {MIN_TITLE_ADS} ads): {len(by_title)}")

# Evidence example, exactly what a dashboard recommendation card will read:
def show_group(name, stats, top_n=5):
    ranked = sorted(stats["skills"].items(), key=lambda kv: -kv[1][1])[:top_n]
    print(f"\n  Higher-paying '{name}' ads: {stats['n_high']:,} — top skills by share:")
    for skill, (count, share) in ranked:
        print(f"    {skill:22} in {share:5.1%} of them ({count:,} ads)")

example_title = "hr manager" if "hr manager" in by_title else next(iter(by_title))
show_group(example_title, by_title[example_title])
show_group("Human Resources & Recruitment", by_category["Human Resources & Recruitment"])
show_group("Information & Communication Technology",
           by_category["Information & Communication Technology"])

# Save as one more dashboard artifact and prove it round-trips
stats_path = "../models/skill_stats.joblib"
joblib.dump({
    "by_category": by_category,
    "by_title": by_title,          # keyed by LOWERCASED job title
    "min_title_ads": MIN_TITLE_ADS,
}, stats_path)
reloaded_stats = joblib.load(stats_path)
print(f"\nSaved {stats_path} ({os.path.getsize(stats_path) / 1024:,.0f} KB) | reload check: "
      f"{len(reloaded_stats['by_category'])} categories, {len(reloaded_stats['by_title'])} titles")

Category groups: 29 | title groups (>= 30 ads): 81

  Higher-paying 'hr manager' ads: 25 — top skills by share:
    communication          in 80.0% of them (20 ads)
    leadership             in 64.0% of them (16 ads)
    payroll                in 52.0% of them (13 ads)
    initiative             in 52.0% of them (13 ads)
    problem solving        in 44.0% of them (11 ads)

  Higher-paying 'Human Resources & Recruitment' ads: 859 — top skills by share:
    communication          in 65.8% of them (565 ads)
    payroll                in 47.6% of them (409 ads)
    interpersonal          in 40.0% of them (344 ads)
    initiative             in 35.3% of them (303 ads)
    organizational         in 28.2% of them (242 ads)

  Higher-paying 'Information & Communication Technology' ads: 1,340 — top skills by share:
    communication          in 59.7% of them (800 ads)
    problem solving        in 35.6% of them (477 ads)
    analytical             in 25.7% of them (344 ads)
    sql           